<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/aiagent.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Day 1 Lab5 - Project</h2>
            <span style="color:#ff7800;">Research Agents<br/>
            This is the fifth lab.<br/>
            Commercial implications.<br/>
            A Deep Research agent is broadly applicable to any business area, and to your own day-to-day activities. You can make use of this yourself!
            </span>
        </td>
    </tr>
</table>

## Deep Research

One of the classic cross-business Agentic use cases! This is huge.

In [1]:
from agents import Agent, WebSearchTool, trace, Runner
from agents.model_settings import ModelSettings
from IPython.display import Markdown, display
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio

In [2]:
load_dotenv(override=True)

True

In [3]:
# Constants 

MODEL_NAME = "gpt-5.4-mini"
HOW_MANY_SEARCHES = 3

## Strategy for the Deep Research Agent

We are going to do it the bulletproof way.

We are going to orchestrate with code: separate calls to `Runner.run()` for each step in the process.

We will use Structured Outputs at each point.

## We will build 5 Agents:

1. The Search Agent: searches the web for information
2. The Planner Agent: given a question, comes up with a list of searches that should be made
3. The fact-checker agent: an agent to check validity of the search.
4. The Writer Agent: writes a robust report
5. The Emailer Agent: crafts and sends an email (we don't cover this part)

And then 5 Python functions, 1 to call Runner.run() for each of the 4 agents.


<table style="margin: 0; text-align: left; width:100%; border-collapse: collapse;">
<tr>
<td rowspan="3" style="width: 220px; text-align: center; vertical-align: middle;">
<img src="../assets/aiagent3.png" width="230" height="200" style="display: block; margin: 0 auto;" />
<h2 style="color:#ff7800; margin: 8px 0 0 0;">Planner Agent</h2>
</td>
<td style="width: 100px; text-align: center; vertical-align: middle; padding: 0 8px;">
<span style="color:#ff7800; font-size: 32px; font-weight: bold; line-height: 1;">→</span>
</td>
<td style="width: 200px; text-align: center; vertical-align: middle; padding: 4px 0;">
<img src="../assets/aiagent3.png" width="90" height="90" style="display: block; margin: 0 auto;" />
<h4 style="color:#ff7800; margin: 4px 0 0 0;">Search Agent 1</h4>
</td>
<td style="width: 100px; text-align: center; vertical-align: middle; padding: 0 8px;">
<span style="color:#ff7800; font-size: 32px; font-weight: bold; line-height: 1;">→</span>
</td>
<td style="width: 200px; text-align: center; vertical-align: middle; padding: 4px 0;">
<img src="../assets/aiagent3.png" width="90" height="90" style="display: block; margin: 0 auto;" />
<h4 style="color:#ff7800; margin: 4px 0 0 0;">Fact-Checker Agent 1</h4>
</td>
<td style="width: 100px; text-align: center; vertical-align: middle; padding: 0 8px;">
<span style="color:#ff7800; font-size: 32px; font-weight: bold; line-height: 1;">→</span>
</td>
<td rowspan="3" style="width: 220px; text-align: center; vertical-align: middle;">
<img src="../assets/aiagent3.png" width="230" height="200" style="display: block; margin: 0 auto;" />
<h2 style="color:#ff7800; margin: 8px 0 0 0;">Writer Agent</h2>
</td>
<td rowspan="3" style="width: 200px; text-align: center; vertical-align: middle; padding: 0 4px;">
<span style="color:#ff7800; font-size: 72px; font-weight: bold; line-height: 1;">→</span>
</td>
<td rowspan="3" style="width: 220px; text-align: center; vertical-align: middle;">
<img src="../assets/aiagent3.png" width="230" height="200" style="display: block; margin: 0 auto;" />
<h2 style="color:#ff7800; margin: 8px 0 0 0;">Emailer Agent</h2>
</td>
</tr>
<tr>
<td style="width: 100px; text-align: center; vertical-align: middle; padding: 0 8px;">
<span style="color:#ff7800; font-size: 32px; font-weight: bold; line-height: 1;">→</span>
</td>
<td style="text-align: center; vertical-align: middle; padding: 4px 0;">
<img src="../assets/aiagent3.png" width="90" height="90" style="display: block; margin: 0 auto;" />
<h4 style="color:#ff7800; margin: 4px 0 0 0;">Search Agent 2</h4>
</td>
<td style="width: 100px; text-align: center; vertical-align: middle; padding: 0 8px;">
<span style="color:#ff7800; font-size: 32px; font-weight: bold; line-height: 1;">→</span>
</td>
<td style="text-align: center; vertical-align: middle; padding: 4px 0;">
<img src="../assets/aiagent3.png" width="90" height="90" style="display: block; margin: 0 auto;" />
<h4 style="color:#ff7800; margin: 4px 0 0 0;">Fact-Checker Agent 2</h4>
</td>
<td style="width: 100px; text-align: center; vertical-align: middle; padding: 0 8px;">
<span style="color:#ff7800; font-size: 32px; font-weight: bold; line-height: 1;">→</span>
</td>
</tr>
<tr>
<td style="width: 100px; text-align: center; vertical-align: middle; padding: 0 8px;">
<span style="color:#ff7800; font-size: 32px; font-weight: bold; line-height: 1;">→</span>
</td>
<td style="text-align: center; vertical-align: middle; padding: 4px 0;">
<img src="../assets/aiagent3.png" width="90" height="90" style="display: block; margin: 0 auto;" />
<h4 style="color:#ff7800; margin: 4px 0 0 0;">Search Agent 3</h4>
</td>
<td style="width: 100px; text-align: center; vertical-align: middle; padding: 0 8px;">
<span style="color:#ff7800; font-size: 32px; font-weight: bold; line-height: 1;">→</span>
</td>
<td style="text-align: center; vertical-align: middle; padding: 4px 0;">
<img src="../assets/aiagent3.png" width="90" height="90" style="display: block; margin: 0 auto;" />
<h4 style="color:#ff7800; margin: 4px 0 0 0;">Fact-Checker Agent 3</h4>
</td>
<td style="width: 100px; text-align: center; vertical-align: middle; padding: 0 8px;">
<span style="color:#ff7800; font-size: 32px; font-weight: bold; line-height: 1;">→</span>
</td>
</tr>
</table>

## Agent 1: The Search Agent

### OpenAI Hosted Tools

https://openai.github.io/openai-agents-python/tools/#hosted-tools

A paid, quick approach to carrying out managed functionality on OpenAI's cloud.

Their docs surface these tools, but it's worth keeping in mind that they're costly and lock you into the OpenAI ecosystem.

OpenAI offers the following hosted tools:

`WebSearchTool` lets an agent search the web.  
`FileSearchTool` allows retrieving information from your OpenAI Vector Stores.  
`CodeInterpreterTool` lets the LLM execute code in a sandboxed environment.  
`HostedMCPTool` exposes a remote MCP server's tools to the model.  
`ImageGenerationTool` generates images from a prompt.  
`ToolSearchTool` lets the model load deferred tools, namespaces, or hosted MCP servers on demand.  

### Important note - API charge of WebSearchTool

This currently costs 1 cent per call for OpenAI WebSearchTool.
Costs are in the Tools section here: https://developers.openai.com/api/docs/pricing


In [10]:
INSTRUCTIONS = """
You are a research assistant. Given a search term, you search the web for that term and 
produce a concise summary of the results. The summary must be 2-3 paragraphs and less than 300 words.
Capture the main points and be succinct. Reply only with the summary.
"""
task = "List the top three wireless (Access Point) vendors in 2026 in terms of prices and services"

settings = ModelSettings(tool_choice="required")
tools = [WebSearchTool()]

In [11]:
search_agent = Agent(name="Search Agent", instructions=INSTRUCTIONS, tools=tools, model=MODEL_NAME, model_settings=settings)

In [ ]:
result = await Runner.run(search_agent, task)
display(Markdown(result.final_output))

### As always, take a look at the trace

https://platform.openai.com/traces

## Agent 2: The Planner Agent

## We will now use Structured Outputs, and include a description of the fields

We bring a reason why the search has to be done!<br>
    reason: str = Field(description="Your reasoning for why this search is important to the query.")<br>
and then the query:<br>
    query: str = Field(description="The search term to use for the web search.")<br>

There's a subtle hidden point here!</br>
We're putting something rationale here, so the model has to generate a reason for why it's doing what it's doing!<br>
It's using tokens to prepare the thought process; it improves performance.<br>
Because of that, it's strangely important to put the reason prior to the query which is the actual search term.

## What will happen if we put the query first and the reason second?<br>
Then the LLM will generate the query, and the reason will simply be something that will be constructed to justify the query after the fact to make for a consistent response.<br>Justifications after the fact aren't as good as having a rationale before the query is generated.

In [13]:
# BaseModel is Pydantic’s class for a typed data shape. It is imported at the top of the lab:
# from pydantic import BaseModel, Field

from pydantic import BaseModel, Field
class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")
    query: str = Field(description="The search term to use for the web search.")

# The WebSearchPlan is a list of WebSearchItem

class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")

In [ ]:
# To show the schema of the WebSearchPlan

WebSearchPlan.model_json_schema()

In [15]:

INSTRUCTIONS = f"""
You are a research assistant. Given a user query, come up with a set of web searches
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for; you also include the references in the end.
"""

planner_agent = Agent(name="Planner Agent", instructions=INSTRUCTIONS, model=MODEL_NAME, output_type=WebSearchPlan)

In [16]:

result = await Runner.run(planner_agent, task)
# result.final_output

## Agent 3: The Fact Checker Agent

Checks **each** cited search summary with a fresh web search immediately after it. It returns **approved**, **partially approved**, or **disapproved**. Partially approved results are corrected so supported evidence can still reach the Writer; only unsalvageable results are dropped.


In [17]:
from typing import Literal

INSTRUCTIONS = """
You are a fact checker. You receive the original research query and ONE cited search-agent
summary. Use a fresh web search to independently evaluate its material claims. Do not write a
new report and do not demand any predetermined publisher or website.

Check rankings, vendor lists, numbers, prices, dates, and service comparisons separately.
Judge each source according to the claim it supports: direct/current evidence is preferable,
but relevant trade publications, distributors, reviews, and other credible sources may be used
when primary evidence is unavailable. Treat transparent uncertainty as a strength. A price may
be usable when accurately labeled as approximate, regional, reseller-based, model-specific,
subscription-dependent, or quote-only.

Verdict rules:
- approved: The main claims are supported and the summary can be used as written. Minor gaps or
  harmless imprecision are acceptable.
- partially_approved: Some claims need removal or correction, but enough verified information
  remains to be useful. Return a corrected verified_summary containing only supported claims.
- disapproved: The central claims are false, contradicted, or too poorly supported to salvage.
  Return an empty verified_summary.

Do not disapprove merely because no single source states the summary's overall synthesis or
ranking verbatim. A synthesis is acceptable when its criteria are clear and its component claims
are supported. When evidence conflicts, preserve the supported facts and explicitly describe the
uncertainty. Cite the sources used in your findings and retain useful citations in verified_summary.
"""

class FactCheckResult(BaseModel):
    findings: str = Field(description="Claim-level verification findings with source citations.")
    verdict: Literal["approved", "partially_approved", "disapproved"] = Field(
        description="Whether the summary is usable as written, usable after correction, or unusable."
    )
    verified_summary: str = Field(
        description="A corrected cited summary containing only verified claims; empty only when disapproved."
    )

fact_checker_agent = Agent(
    name="Fact Checker Agent",
    instructions=INSTRUCTIONS,
    model=MODEL_NAME,
    tools=[WebSearchTool()],
    model_settings=ModelSettings(tool_choice="required"),
    output_type=FactCheckResult,
)


## Agent 4: The Writer Agent

In [18]:
INSTRUCTIONS = """
You are a senior researcher tasked with writing a cohesive report for a research query.
You will be provided with the original query, and some research.
Generate a comprehensive report based on the research and the query.
The final output should be in markdown format, and it should be lengthy and detailed. Aim 
for 5-10 pages of content, at least 1000 words.
"""


class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")
    markdown_report: str = Field(description="The final report")
    follow_up_questions: list[str] = Field(description="Suggested topics to research further")


writer_agent = Agent(name="Writer Agent", instructions=INSTRUCTIONS, model=MODEL_NAME, output_type=ReportData)

## Agent 5: The Email Agent

#### We don't cover this section in this lab.

## Now to Orchestrate by Code

The next functions plan complementary searches, run each search, then fact-check it immediately. Approved summaries and corrected portions of partially approved summaries are retained.

In [19]:
async def run_searches(query: str):
    print("Planning searches...")
    result = await Runner.run(planner_agent, f"Query: {query}")
    searches = result.final_output.searches
    print(f"Will perform {len(searches)} searches")
    tasks = [search_and_check(query, item) for item in searches]
    results = await asyncio.gather(*tasks)
    usable = [summary for summary in results if summary is not None]
    print(f"Finished searching: {len(usable)} of {len(searches)} usable")
    return usable


async def search_and_check(query: str, item: WebSearchItem):
    input_message = f"Search term: {item.query}\nReason for searching: {item.reason}"
    result = await Runner.run(search_agent, input_message)
    summary = result.final_output
    fact_check = await check_facts(query, item, summary)
    if fact_check.verdict == "disapproved":
        print(f"Discarding unsalvageable search: {item.query}")
        return None
    if fact_check.verdict == "partially_approved":
        print(f"Using corrected summary for: {item.query}")
    return fact_check.verified_summary or summary

Fact-check one search at a time. Write the report from approved evidence and corrected, verified portions of partially approved searches.

In [20]:
async def check_facts(query: str, item: WebSearchItem, summary: str):
    print(f"Fact-checking search: {item.query}")
    input_message = (
        f"Original query: {query}\n"
        f"Search term: {item.query}\n"
        f"Reason for searching: {item.reason}\n"
        f"Search summary to verify:\n{summary}"
    )
    result = await Runner.run(fact_checker_agent, input_message)
    fact_check = result.final_output
    print(f"Fact check for '{item.query}': {fact_check.verdict}")
    print(f"Findings: {fact_check.findings}\n")
    return fact_check

    
async def write_report(query: str, search_results: list[str]):
    print("Thinking about report...")
    joined_results = "\n\n--- FACT-CHECKED RESULT ---\n".join(search_results)
    input_message = f"Original query: {query}\n\nFact-checked research:\n{joined_results}"
    result = await Runner.run(writer_agent, input_message)
    print("Finished writing report")
    return result.final_output


# the email agent will appear here


### Showtime!

In [ ]:
query = """
Compare three leading enterprise wireless access-point vendors in 2026. Select candidates based
on evidence of market presence, then compare representative hardware pricing or pricing
transparency, technology, and their support, cloud-management, and related service offerings.
Explain the selection methodology, avoid claiming a universal ranking, and identify material
uncertainty.
"""

with trace("Research trace"):
    print("Starting research...")
    search_results = await run_searches(query)
    if not search_results:
        print("No usable evidence remained after fact-checking. No report will be written.")
        display(Markdown("**No report:** no search contained evidence the fact checker could safely retain."))
    else:
        report = await write_report(query, search_results)
        display(Markdown(report.markdown_report))


### As always, take a look at the trace

https://platform.openai.com/traces